In [67]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import Perceptron
from sklearn.metrics import (accuracy_score,roc_auc_score,f1_score,confusion_matrix,classification_report)
from sklearn.metrics import (roc_auc_score,precision_score, recall_score, f1_score)
import time
import traceback
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
RANDOM_STATE = 0

In [68]:
df=pd.read_csv(r"C:\Users\user\Downloads\archive (1).zip")
df.shape

(193852, 2)

In [69]:
df.columns

Index(['label', 'text'], dtype='object')

In [70]:
df.head()

,label,text
0,Spam,viiiiiiagraaaa\nonly for the ones that want to...
1,Ham,got ice thought look az original message ice o...
2,Spam,yo ur wom an ne eds an escapenumber in ch ma n...
3,Spam,start increasing your odds of success & live s...
4,Ham,author jra date escapenumber escapenumber esca...


In [71]:
df.describe()

,label,text
count,193852,193850
unique,2,193848
top,Ham,unsubscribe
freq,102160,2


In [72]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 193852 entries, 0 to 193851
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   label   193852 non-null  object
 1   text    193850 non-null  object
dtypes: object(2)
memory usage: 3.0+ MB


In [73]:
df['text']=df['text'].fillna("")

In [74]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 193852 entries, 0 to 193851
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   label   193852 non-null  object
 1   text    193852 non-null  object
dtypes: object(2)
memory usage: 3.0+ MB


In [75]:
df['label']=df['label'].map({"Spam":1,"Ham":0})

In [76]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 193852 entries, 0 to 193851
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   label   193852 non-null  int64 
 1   text    193852 non-null  object
dtypes: int64(1), object(1)
memory usage: 3.0+ MB


In [77]:
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
x = vectorizer.fit_transform(df['text'])

In [78]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 193852 entries, 0 to 193851
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   label   193852 non-null  int64 
 1   text    193852 non-null  object
dtypes: int64(1), object(1)
memory usage: 3.0+ MB


In [79]:
df.isnull().sum()

label    0
text     0
dtype: int64

In [80]:
y=df['label']
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=0,
    stratify=y
)

print(f"Train size: {x_train.shape[0]:,}  Test size: {x_test.shape[0]:,}")

Train size: 155,081  Test size: 38,771


In [81]:
base_models = [
    ("Logistic Regression", LogisticRegression(max_iter=1000)),
    ("Perceptron", Perceptron(random_state=0)),
    ("Decision Tree", DecisionTreeClassifier(random_state=0, max_depth=15))
]

In [82]:
meta_model = LogisticRegression(max_iter=1000)

In [83]:
meta_train = np.zeros((x_train.shape[0], len(base_models)))
meta_test = np.zeros((x_test.shape[0], len(base_models)))

In [84]:
for column, (model_name, model) in enumerate(base_models):
    print(model_name)
    # Out-of-fold predictions
    oof_predictions = cross_val_predict(model,x_train,y_train,cv=kfold,method="predict")
    meta_train[:, column] = oof_predictions

Logistic Regression
Perceptron
Decision Tree


In [85]:
 #Train on entire training set
model.fit(x_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,15
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [86]:
# TRAIN META MODEL
meta_model.fit(meta_train, y_train)

stack_pred = meta_model.predict(meta_test)

In [88]:
results = []

for model_name, model in base_models:
    model.fit(x_train, y_train)
    
    pred = model.predict(x_test)
    
    auc = roc_auc_score(y_test, pred)
    precision = precision_score(y_test, pred, zero_division=0)
    recall = recall_score(y_test, pred, zero_division=0)
    f1 = f1_score(y_test, pred, zero_division=0)

    results.append([model_name, auc, precision, recall, f1])


In [90]:
for column, (model_name, model) in enumerate(base_models):
    pred = model.predict(x_test)
    meta_test[:, column] = pred

stack_pred = meta_model.predict(meta_test)

stack_auc = roc_auc_score(y_test, stack_pred)
stack_precision = precision_score(y_test, stack_pred, zero_division=0)
stack_recall = recall_score(y_test, stack_pred, zero_division=0)
stack_f1 = f1_score(y_test, stack_pred, zero_division=0)


results.append([ "Stacking",stack_auc,stack_precision,stack_recall,stack_f1])


In [91]:
# Show all results
results_df = pd.DataFrame(
    results,
    columns=["Model", "AUC", "Precision", "Recall", "F1"]
)

results_df

,Model,AUC,Precision,Recall,F1
0,Logistic Regression,0.973094,0.967704,0.975408,0.971540
1,Perceptron,0.966603,0.956522,0.972899,0.964641
2,Decision Tree,0.912208,0.854945,0.972518,0.909949
3,Stacking,0.975251,0.961944,0.985495,0.973577
